In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
import numpy.fft as fft

In [ ]:
criteria =  ["coupling=1.0", "vx=0.05", "Nx=4096", "Ny=1024"]
data_files = sorted(Path("./data2d").glob("*.npz"))
data_files = list(filter(lambda f: all(criterion in f.name for criterion in criteria), data_files))

In [ ]:
#df = []
#for data_file in tqdm(data_files):
#    data = np.load(data_file, allow_pickle=True)
#    params = data["params"].item()
#    if "v" in params:
#        params["vx"] = params["v"][0]
#        params["vy"] = params["v"][1]
#        del params["v"]
#    z_hist = data["z_hist"]
#    phi_q_hist = data["phi_qs"]
#    df.append(params | {"tseries": z_hist, "phi_qseries": phi_q_hist})
#df = pd.DataFrame(df)

In [ ]:
for data_file in tqdm(data_files):
    data = np.load(data_file, allow_pickle=True)
    params = data["params"].item()
    if "v" in params:
        params["vx"] = params["v"][0]
        params["vy"] = params["v"][1]
        del params["v"]
    z_hist = data["z_hist"]
    x_hist = np.copy(z_hist)
    x_hist[:, 0] += params["dt"]*params["vx"]*np.arange(z_hist.shape[0])
    x_hist[:, 1] += params["dt"]*params["vy"]*np.arange(z_hist.shape[0])
    x_hist_wrapped = np.copy(x_hist)
    x_hist_wrapped[:, 0] = x_hist_wrapped[:, 0] % (params["dx"] * params["Nx"])
    x_hist_wrapped[:, 1] = x_hist_wrapped[:, 1] % (params["dx"] * params["Ny"])
    phi_q_hist = data["phi_qs"]
    phi_x_hist = np.fft.ifft2(phi_q_hist, axes=(-2, -1)).real
    phi_time_index = np.arange(0, z_hist.shape[0], z_hist.shape[0] // phi_x_hist.shape[0])
    break

In [ ]:
phi_min, phi_max = np.quantile(phi_x_hist, [0.01, 0.99])
for phi_idx in range(0,phi_x_hist.shape[0],10):
    plt.figure(figsize=(4, 4))
    x_coords = params["dx"] * np.arange(phi_x_hist.shape[-2])
    y_coords = params["dx"] * np.arange(phi_x_hist.shape[-1])
    plt.contourf(x_coords, y_coords, phi_x_hist[phi_idx].T, levels=100, cmap="RdBu_r", vmin=phi_min, vmax=phi_max)
    ax = plt.gca()
    ax.set_aspect(1)
    ax.scatter(x_hist_wrapped[phi_time_index[phi_idx], 0], params["Ly"]/2, color="black", s=10)
    ax.scatter(params["vx"]*params["dt"]*phi_time_index[phi_idx] % params["Lx"], params["Ly"]/2, color="green", s=10)
    plt.show()

In [ ]:
phi_min, phi_max = phi_x_hist.min(), phi_x_hist.max()
for phi_idx in range(0,phi_x_hist.shape[0],100):
    plt.figure(figsize=(4, 4))
    x_coords = params["dx"] * np.arange(phi_x_hist.shape[-2])
    y_coords = params["dx"] * np.arange(phi_x_hist.shape[-1])
    plt.plot( x_coords, phi_x_hist[phi_idx, :, :].mean(axis=1))
    ax = plt.gca()
    ax.set_aspect(1e3)
    x_idx = np.argmin(np.abs(x_coords - x_hist_wrapped[phi_time_index[phi_idx], 0]))
    ax.scatter(x_hist_wrapped[phi_time_index[phi_idx], 0], phi_x_hist[phi_idx, x_idx, :].mean(), color="red", s=100)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$\\phi(x)$")
    ax.set_ylim(phi_min, phi_max)
    plt.show()

In [ ]:
start_time = 300
end_time = start_time + 10
plot_traces = True
for coupling, df_coupling in df.groupby("coupling"):
    num_plots = len(df_coupling)
    df_coupling = df_coupling.sort_values("coupling")
    vels = df_coupling["vx"].values
    fig, axes = plt.subplots(1, num_plots, figsize=(6*num_plots, 4), sharex=plot_traces)
    if num_plots == 1:
        axes = [axes]
    for i, (idx, row) in enumerate(df_coupling.iterrows()):
        tseries = row["tseries"]
        
        time = np.arange(len(tseries))*row["dt"]
        mask = (time >= start_time) & (time < end_time)
        
        x = tseries[mask,0]
        t = time[mask]
        
        if(plot_traces):
            axes[i].plot(t, x, label="x")
        else:
            axes[i].hist(x, bins=100, density=True)
            axes[i].set_yscale("log")
        axes[i].set_title(f"vx={row['vx']:.2f}, vy={row['vy']:.2f}")
    fig.suptitle(f"coupling={row['coupling']} ")
        
    plt.show()